<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 03 - Decision Tree: validación cruzada

En este notebook se evalúa el modelo **Decision Tree** mediante validación cruzada.

La validación cruzada permite obtener una estimación más estable del rendimiento del modelo, ya que no depende únicamente de una división concreta entre entrenamiento y test.

## Objetivo metodologico

La validacion cruzada complementa la evaluacion con una unica particion train/test. Al entrenar y evaluar en varias particiones, se obtiene una estimacion mas estable del rendimiento esperado del arbol.


## 1. Importación de librerías y configuración inicial

Se cargan las librerías necesarias, el modelo Decision Tree y las rutas del proyecto.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import cross_val_score

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.model_training import get_base_models

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Validación cruzada del modelo

Se usa `cross_val_score` con 5 particiones y la métrica `f1`, adecuada para una tarea de clasificación binaria.

### Lectura de resultados

`mean_f1` resume el rendimiento medio y `std_f1` indica estabilidad. Una desviacion alta significa que el modelo depende demasiado de que ejemplos caen en cada particion.


In [2]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
X = df.drop(columns=["new_id", "mature"])
y = df["mature"]
model = get_base_models(random_state=42)["decision_tree"]
scores = cross_val_score(model, X, y, scoring="f1", cv=5, n_jobs=-1)
cv_metrics = pd.DataFrame([{
    "model": "decision_tree",
    "mean_f1": scores.mean(),
    "std_f1": scores.std(),
    "fold_scores": ",".join(f"{score:.4f}" for score in scores)
}])
cv_metrics

,model,mean_f1,std_f1,fold_scores
0,decision_tree,0.366522,0.017173,"0.3461,0.3766,0.3904,0.3718,0.3477"


## 3. Guardado de métricas de validación cruzada

Se guardan los resultados medios y la desviación típica para incluirlos en la comparativa final.

In [3]:
cv_metrics.to_csv(RESULTS_DIR / "dt_cv_metrics.csv", index=False)

## 4. Conclusiones

La validación cruzada permite comprobar si el rendimiento del árbol de decisión es estable o si depende demasiado de una división concreta de los datos.